# GMM clustering notebook
Run notebooks in the following order:\
`preprocessing.ipynb -> imputation.R -> dimensionality_reduction.R -> gmm_clustering.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path(os.getcwd()), *Path(os.getcwd()).parents] if path.stem == "esa_consulting")
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
import plotnine as p9
from gapstat import gapstat_score
from IPython.display import display
from scipy.stats import chi2_contingency
from sklearn import clone

from utils.bootstrap_utils import bootstrap, bootstrap_grid_search
from utils.clustering_utils import (
    cluster_summaries,
    predict_clustering_labels,
    predict_clustering_proba,
    print_cluster_stability,
    print_cluster_statistics,
    print_clustering_info,
)
from utils.gmm_utils import GaussianMixture
from utils.metrics_utils import (
    adjusted_rand_score,
    association,
    calinski_harabasz_score,
    davies_bouldin_score,
    dunn_index,
    normalized_mutual_info_score,
    silhouette_score,
    variation_of_information,
)
from utils.misc_utils import contingency_matrix, std_weighted, turn_cats_to_continuous
from utils.plotting_utils import make_plot_mixture_weights, theme_ggplot
from utils.saving_utils import DatasetBundle, save_plot

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Load data
data_discov_full = DatasetBundle.load(Path("data/preprocessed/data_discov_full"))
data_discov_full.meta = cast(pd.DataFrame, data_discov_full.meta)
data_discov_full.na = cast(pd.DataFrame, data_discov_full.na)
data_valid_full = DatasetBundle.load(Path("data/preprocessed/data_valid_full"))
data_valid_full.meta = cast(pd.DataFrame, data_valid_full.meta)
data_valid_full.na = cast(pd.DataFrame, data_valid_full.na)

data_discov_mfa = pd.read_parquet(Path("data/dim_red/data_discov_mfa.parquet"))
data_valid_mfa = pd.read_parquet(Path("data/dim_red/data_valid_mfa.parquet"))

# Set outputs folder
outputs_folder = "outputs/gmm_clustering"

# Set sample weights
sample_weight_discov = data_discov_full.data["wichtn"].to_numpy()
sample_weight_valid = data_valid_full.data["wichtn"].to_numpy()

In [ ]:
# Create RNG for reproducibility
rng = np.random.default_rng(352532)
rng1, rng2, rng3 = rng.spawn(3)
gmm_discov = GaussianMixture(random_state=np.random.RandomState(rng1.bit_generator))
gmm_valid = GaussianMixture(random_state=np.random.RandomState(rng2.bit_generator))

n_boot = 10
param_grid = [
    # {"n_components": range(2, 51), "covariance_type": ["spherical"]},
    # {"n_components": range(2, 51), "covariance_type": ["tied"]},
    # {"n_components": range(2, 51), "covariance_type": ["diag"]},
    {"n_components": range(2, 51), "covariance_type": ["full"]},
]
within_metrics = (
    lambda e, X, sample_weight: e.score(X, sample_weight=sample_weight),
    lambda e, X, sample_weight: e.bic(X, sample_weight=sample_weight),
    lambda e, X, sample_weight: e.icl(X, sample_weight=sample_weight),
    lambda e, X, sample_weight: e.aic(X, sample_weight=sample_weight),
    lambda e, X, sample_weight: davies_bouldin_score(X, labels=e.predict(X), sample_weight=sample_weight),
    lambda e, X, sample_weight: calinski_harabasz_score(X, labels=e.predict(X), sample_weight=sample_weight),
    lambda e, X, sample_weight: silhouette_score(X, labels=e.predict(X), sample_weight=sample_weight),
    lambda e, X, sample_weight: dunn_index(X, labels=e.predict(X)),
)
between_metrics = (
    lambda x, y, sample_weight: adjusted_rand_score(x, y, sample_weight=sample_weight),
    lambda x, y, sample_weight: normalized_mutual_info_score(x, y, sample_weight=sample_weight),
    lambda x, y, sample_weight: variation_of_information(x, y, sample_weight=sample_weight),
    lambda x, y, sample_weight: association(contingency_matrix(x, y, sample_weight=sample_weight), correction=False),
    lambda x, y, sample_weight: (
        chi2_contingency(contingency_matrix(x, y, sample_weight=sample_weight), correction=False).statistic
    ),
)
within_metric_names = {i: name for i, name in enumerate(["loglike", "bic", "icl", "aic", "dbi", "chi", "si", "di"])}
between_metric_names = {i: name for i, name in enumerate(["ari", "nmi", "vi", "assoc", "chi2"])}

within_results_discov, between_results_discov = bootstrap_grid_search(
    gmm_discov,
    data_discov_mfa.to_numpy(),
    param_grid,
    n_boot=n_boot,
    within_metrics=within_metrics,
    between_metrics=between_metrics,
    sample_weight=sample_weight_discov,
    rng=rng3,
)

within_results_discov["metric"] = within_results_discov["metric"].cat.rename_categories(within_metric_names)
between_results_discov["metric"] = between_results_discov["metric"].cat.rename_categories(between_metric_names)
metrics_results_discov = pd.concat([within_results_discov, between_results_discov], ignore_index=True)

In [ ]:
# Plot all metrics
(
    p9.ggplot(metrics_results_discov)
    + p9.aes(x="n_components", y="mean")  # , group="covariance_type")
    + p9.geom_ribbon(p9.aes(ymin="mean - std", ymax="mean + std"), alpha=0.2)
    + p9.geom_line()  # p9.aes(color="covariance_type"))
    + p9.facet_wrap("metric", scales="free_y")
    + theme_ggplot
)

In [ ]:
# Plot a single metric
metric = "icl"
(
    p9.ggplot(metrics_results_discov[lambda x: x["metric"].eq(metric)])
    + p9.aes(x="n_components", y="mean")  # , group="covariance_type")
    + p9.geom_ribbon(p9.aes(ymin="mean - std", ymax="mean + std"), alpha=0.2)
    + p9.geom_line()  # p9.aes(color="covariance_type"))
    + theme_ggplot
)

In [ ]:
# Find best performing parameters
smaller_is_better_metrics = ["loglike", "bic", "icl", "aic", "dbi", "vi"]
bigger_is_better_metrics = ["chi", "si", "di", "ari", "nmi", "vi", "assoc", "chi2"]
min_metrics = metrics_results_discov.loc[lambda x: x["metric"].isin(smaller_is_better_metrics)].loc[
    lambda x: x.groupby("metric")["mean"].idxmin()
]
max_metrics = metrics_results_discov.loc[lambda x: x["metric"].isin(bigger_is_better_metrics)].loc[
    lambda x: x.groupby("metric")["mean"].idxmax()
]
best_metrics = pd.concat([min_metrics, max_metrics])
print("minimum mean values:")
display(min_metrics)
print("maximum mean values:")
display(max_metrics)

In [ ]:
# Set best parameters and refit models
selection_metric = "icl"
best_params = (
    best_metrics[lambda x: x["metric"].eq(selection_metric)]
    .drop(columns=["metric", "mean", "std"])
    .to_dict(orient="records")[0]
)
print(f"Best parameters wrt. {selection_metric}: {best_params}")

gmm_discov.set_params(**best_params)
gmm_valid.set_params(**best_params)
model_discov = clone(gmm_discov)
model_valid = clone(gmm_valid)
model_discov.fit(data_discov_mfa, sample_weight=sample_weight_discov)
model_valid.fit(data_valid_mfa, sample_weight=sample_weight_valid)
None

In [ ]:
# Apply models to predict labels
labels_discov = predict_clustering_labels(model_discov, data_discov_mfa)
labels_valid = predict_clustering_labels(model_valid, data_valid_mfa)

# Apply models to predict probabilities
proba_discov = predict_clustering_proba(model_discov, data_discov_mfa)
proba_valid = predict_clustering_proba(model_valid, data_valid_mfa)

# Save predictions
proba_discov.assign(**{"label": labels_discov}).to_parquet(
    Path(outputs_folder, "cluster_pred_discov.parquet"), index=False
)
proba_valid.assign(**{"label": labels_valid}).to_parquet(
    Path(outputs_folder, "cluster_pred_valid.parquet"), index=False
)

### Display model information

In [ ]:
# Print information about clusterings
print("Discovery:    ", end="")
print_clustering_info(labels_discov)
print("Validation:   ", end="")
print_clustering_info(labels_valid)

# Plot mixture weights
plot_discov_weights = make_plot_mixture_weights(model_discov)
plot_valid_weights = make_plot_mixture_weights(model_valid)

# save_plot(plot_discov_weights, Path(outputs_folder, "plot_discov_weights.png"), dpi=150)
# save_plot(plot_valid_weights, Path(outputs_folder, "plot_valid_weights.png"), dpi=150)

# Combine the two plots
plot_discov_weights += p9.labs(title="Discovery")
plot_valid_weights += p9.labs(title="Validation")
plot_all_weights = (plot_discov_weights / plot_valid_weights) & p9.theme(figure_size=(5, 5))

plot_all_weights.show()

save_plot(plot_all_weights, Path(outputs_folder, "plot_all_weights.png"), dpi=150)

### Get cluster summaries

In [ ]:
treat_cat_as_continuous = data_discov_full.meta.loc[lambda x: x["type"].isin(["binary", "ordinal"]), "name"]
data_discov_cont = turn_cats_to_continuous(data_discov_full.data, treat_cat_as_continuous).select_dtypes(
    exclude="category"
)
data_valid_cont = turn_cats_to_continuous(data_valid_full.data, treat_cat_as_continuous).select_dtypes(
    exclude="category"
)

In [ ]:
# Interpretation of continuous, binary, and ordinal variables
cluster_discov_summaries = cluster_summaries(data_discov_cont, labels_discov, sample_weight=sample_weight_discov)
cluster_discov_summaries

In [ ]:
# Interpretation of continuous, binary, and ordinal variables
cluster_valid_summaries = cluster_summaries(data_valid_cont, labels_valid, sample_weight=sample_weight_valid)
cluster_valid_summaries

### Compare clusterings
We are dealing with _inferential clustering_, because we are clustering individuals and not variables.\
Method-based validation: The same clustering method is applied to the validation data, and the results are compared between clusterings.\
Result-based validation: The trained cluster algorithm is applied to the validation data, and the results are compared between datasets.\
Cluster stability: The results from the trained and transferred clusterings from the method- and result-based validation steps are compared.

In [ ]:
# Compare clusterings
within_results_valid, between_results_valid = bootstrap(
    gmm_valid,
    data_valid_mfa,
    n_boot=n_boot,
    within_metrics=within_metrics,
    between_metrics=between_metrics,
    sample_weight=sample_weight_valid,
    rng=np.random.default_rng(2352),
)

within_results_valid["metric"] = within_results_valid["metric"].cat.rename_categories(within_metric_names)
between_results_valid["metric"] = between_results_valid["metric"].cat.rename_categories(between_metric_names)
metrics_results_valid = pd.concat([within_results_valid, between_results_valid], ignore_index=True)

In [ ]:
# Compare clusterings
mask = np.logical_and.reduce([metrics_results_discov[col] == val for col, val in best_params.items()])
(
    metrics_results_discov[mask]
    .drop(columns=best_params)
    .set_index("metric")
    .join(metrics_results_valid.set_index("metric"), lsuffix="_discov", rsuffix="_valid")
    .reset_index()
)